# Evaluation — exact match per model and prompting strategy

Precision / Recall / F1 for exact match, computed key by key over the annotation JSON
and micro-averaged over **all** the notes of each (model, strategy).

* one **slot** = one key: the scalar top-level keys plus the attributes of each medication;
* **TP** = gold has a value and the prediction reproduces it exactly;
  **FP** = the prediction has a value that is not the gold one;
  **FN** = gold has a value that the prediction does not reproduce;
* `null` on both sides is a true negative and is not counted.

**Medications are aligned before being compared** (section 4). Comparing them position
by position makes a single missed medication cascade into a false positive *and* a false
negative on every attribute of every medication after it, which measures list alignment
rather than extraction quality. The positional variant is kept in section 7 as a lower
bound.

**Failed extractions** (files under `<model>/errors/<strategy>/`) are loaded like any
other and marked `status="error"`. In the headline table they count as *no output*:
every annotated slot of that note becomes a false negative. This penalises recall and
never rewards precision, so a model that crashes on the hard notes cannot look better
than one that answers them badly. The `invalid_rate` column and the valid-only table in
section 7 report the other half of the picture.

The sample is a pilot (32 notes), so every headline number carries a bootstrap
confidence interval over notes, and strategies are compared with a **paired** bootstrap
on the same notes (section 8). Differences smaller than the interval are not results.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
ROOT = next(p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / "src").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import json
import re
import shutil
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from clinical_notes_extraction.utils.llm.evaluation import (  # noqa: E402
    ATTRIBUTES,
    TOP_LEVEL_KEYS,
    load_ground_truth_dir,
    load_results_tree,
    metrics_to_docx,
    metrics_to_html,
    metrics_to_latex,  # noqa: F401  (kept for the optional LaTeX export at the end)
)

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("project root:", ROOT)

## 1. Paths

`RUN_ID` pins one extraction run; never mix timestamps in the same table. The run's
`config.json` is copied next to the metrics so the numbers stay tied to the
configuration that produced them (`num_ctx`, `temperature`, `save_prompts`, ...).

In [ ]:
DATA_DIR = NOTEBOOK_DIR / "data"

SPLIT = "dev"                 # ground_truth/<SPLIT>  — check the folder name on disk
CRITERION = "exact_match"

GROUND_TRUTH_DIR = DATA_DIR / "annotations" / "ground_truth" / SPLIT
RESULTS_ROOT = DATA_DIR / "llm_extraction_results" / SPLIT
RUN_ID = "20260812_203855"
RUN_DIR = RESULTS_ROOT / RUN_ID
# RUN_ID = sorted(p.name for p in RESULTS_ROOT.iterdir() if p.is_dir())[-1]  # latest run

OUTPUT_DIR = RESULTS_ROOT / "evaluation" / RUN_ID / CRITERION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert GROUND_TRUTH_DIR.is_dir(), GROUND_TRUTH_DIR
assert RUN_DIR.is_dir(), RUN_DIR

RUN_CONFIG = RUN_DIR / "config.json"
if RUN_CONFIG.is_file():
    shutil.copy(RUN_CONFIG, OUTPUT_DIR / "run_config.json")
    print(json.loads(RUN_CONFIG.read_text(encoding="utf-8")))
else:
    print("no config.json in the run directory — the metrics are not tied to a config")

SEED = 20260814
N_BOOTSTRAP = 2000

## 2. Load

`load_ground_truth_dir` reads one annotation per file (the note id comes from the
`note_id` key, falling back to the file name). `load_results_tree` walks
`<run>/<model>/<strategy>/*.json`, skips `prompts/`, and picks up
`<model>/errors/<strategy>/*.json` as failed records.

In [ ]:
gold_by_id = load_ground_truth_dir(GROUND_TRUTH_DIR)
records = load_results_tree(RUN_DIR)

print(f"{len(gold_by_id)} annotated notes in {SPLIT}")
print(f"{len(records)} result files in run {RUN_ID}")

## 3. Coverage check

Before reading any metric: does every (model, strategy) cover every annotated note?
`missing` are notes with neither an output file nor an error file — they are also
scored as no output, but they usually mean the grid did not finish.

In [ ]:
records_df = pd.DataFrame(records)
annotated = set(gold_by_id)

coverage = (
    records_df[records_df.note_id.isin(annotated)]
    .groupby(["model", "strategy"])
    .agg(ok=("status", lambda s: (s == "ok").sum()),
         errors=("status", lambda s: (s != "ok").sum()))
)
coverage["missing"] = len(annotated) - coverage.ok - coverage.errors
coverage["not_annotated"] = (
    records_df[~records_df.note_id.isin(annotated)]
    .groupby(["model", "strategy"]).size()
)
coverage = coverage.fillna(0).astype(int)
coverage

## 4. Scoring

### 4.1 Canonical form of a value

Everything the comparison does to a value happens here, so that the thesis can state
the rule in one paragraph:

* `None`, `""` and `[]` are all **absent**; absent on both sides is a true negative;
* booleans are canonicalised to `"true"` / `"false"` — `false` is a **value**, not an
  absence (this is what `flag_is_medication_completed` needs);
* `indication` is an array: it is compared as a **set** of canonical strings, so the
  order the model happened to emit is not an error;
* strings are stripped; **nothing else is normalised by default** — the annotation rule
  is verbatim, so casing and internal punctuation are part of the value;
* `casefold=True` and `collapse_ws=True` are the two relaxations used in section 7 to
  split formatting error from extraction error.

`medications_text` is a long verbatim span whose exact match is effectively binary and
dominated by whitespace; it is always whitespace-collapsed and is reported **outside**
the headline micro-average, in its own row.

In [ ]:
ATTRIBUTES = list(ATTRIBUTES)
TOP_LEVEL_KEYS = list(TOP_LEVEL_KEYS)

SPAN_KEY = "medications_text"                       # scored separately
SET_KEYS = {"indication"}                           # arrays compared as sets
HEADLINE_TOP_LEVEL = [k for k in TOP_LEVEL_KEYS if k != SPAN_KEY]
SCORED_KEYS = HEADLINE_TOP_LEVEL + ATTRIBUTES       # what the headline micro sums over
ALL_KEYS = ([SPAN_KEY] if SPAN_KEY in TOP_LEVEL_KEYS else []) + SCORED_KEYS

_WS = re.compile(r"\s+")


def canon(key, value, *, casefold=False, collapse_ws=False):
    """Canonical form of one annotation value, or None when the value is absent."""
    if value is None:
        return None
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (list, tuple, set)):
        items = {canon(key, v, casefold=casefold, collapse_ws=collapse_ws) for v in value}
        items.discard(None)
        return frozenset(items) or None
    if isinstance(value, (int, float)):
        value = repr(value)
    if not isinstance(value, str):
        value = str(value)
    text = value.strip()
    if collapse_ws or key == SPAN_KEY:
        text = _WS.sub(" ", text)
    if casefold:
        text = text.casefold()
    return text or None


def get_attr(medication, key):
    if not isinstance(medication, dict):
        return None
    if key in medication:
        return medication.get(key)
    return (medication.get("attributes") or {}).get(key)

### 4.2 Aligning the medications

Gold and predicted medications are matched by name and span before their attributes are
compared. Similarity is the Jaccard overlap of the token sets of
`active_substance + commercial_name + span_text`, casefolded. Pairs below
`MATCH_THRESHOLD` are not matched: an unmatched gold medication contributes a false
negative on each of its annotated slots, an unmatched predicted medication a false
positive on each of its non-null slots.

The optimal assignment is computed with `scipy.optimize.linear_sum_assignment` when
scipy is available, and greedily otherwise; the two agree on lists this short.

In [ ]:
MATCH_THRESHOLD = 0.20
NAME_KEYS = ("active_substance", "commercial_name")

try:
    from scipy.optimize import linear_sum_assignment
except ImportError:  # pragma: no cover
    linear_sum_assignment = None


def _tokens(medication):
    parts = [get_attr(medication, k) for k in NAME_KEYS]
    parts.append((medication or {}).get("span_text"))
    text = " ".join(str(p) for p in parts if p)
    return set(re.findall(r"[a-z0-9]+", text.casefold()))


def similarity(gold_med, pred_med):
    g, p = _tokens(gold_med), _tokens(pred_med)
    if not g or not p:
        return 0.0
    return len(g & p) / len(g | p)


def align(gold_meds, pred_meds, *, positional=False):
    """Return (pairs, unmatched_gold_idx, unmatched_pred_idx)."""
    if positional:
        n = min(len(gold_meds), len(pred_meds))
        return ([(i, i) for i in range(n)],
                list(range(n, len(gold_meds))),
                list(range(n, len(pred_meds))))

    if not gold_meds or not pred_meds:
        return [], list(range(len(gold_meds))), list(range(len(pred_meds)))

    sim = np.array([[similarity(g, p) for p in pred_meds] for g in gold_meds])
    pairs = []
    if linear_sum_assignment is not None:
        rows, cols = linear_sum_assignment(-sim)
        pairs = [(int(i), int(j)) for i, j in zip(rows, cols) if sim[i, j] >= MATCH_THRESHOLD]
    else:
        taken_g, taken_p = set(), set()
        order = np.dstack(np.unravel_index(np.argsort(-sim, axis=None), sim.shape))[0]
        for i, j in order:
            i, j = int(i), int(j)
            if sim[i, j] < MATCH_THRESHOLD:
                break
            if i not in taken_g and j not in taken_p:
                pairs.append((i, j))
                taken_g.add(i)
                taken_p.add(j)
    matched_g = {i for i, _ in pairs}
    matched_p = {j for _, j in pairs}
    return (sorted(pairs),
            [i for i in range(len(gold_meds)) if i not in matched_g],
            [j for j in range(len(pred_meds)) if j not in matched_p])

### 4.3 One note

`score_note` returns the per-key TP/FP/FN counts, the medication-detection counts, and
one row per disagreeing slot with an error category. The categories are what feeds the
failure-mode analysis: they say *how* the value is wrong, not only that it is.

In [ ]:
_PUNCT = re.compile(r"[^a-z0-9]+")


def _as_text(value):
    if value is None:
        return None
    if isinstance(value, frozenset):
        return " | ".join(sorted(value))
    return str(value)


def categorise(gold_value, pred_value):
    g, p = _as_text(gold_value), _as_text(pred_value)
    if g is None:
        return "hallucinated"
    if p is None:
        return "missed"
    if g.casefold() == p.casefold():
        return "casing"
    gs, ps = _PUNCT.sub("", g.casefold()), _PUNCT.sub("", p.casefold())
    if gs == ps:
        return "formatting"
    if gs and ps and (gs in ps or ps in gs):
        return "span_boundary"
    return "different_value"


def _compare(key, gold_value, pred_value, *, medication, rows, counts, opts):
    g = canon(key, gold_value, **opts)
    p = canon(key, pred_value, **opts)
    if g is None and p is None:
        return
    if g == p:
        counts[key]["tp"] += 1
        return
    if g is not None:
        counts[key]["fn"] += 1
    if p is not None:
        counts[key]["fp"] += 1
    rows.append({
        "medication": medication, "key": key,
        "gold": gold_value, "predicted": pred_value,
        "category": categorise(g, p),
    })


def score_note(gold, prediction, *, positional=False, **opts):
    counts = defaultdict(Counter)
    rows = []
    prediction = prediction or {}

    for key in TOP_LEVEL_KEYS:
        _compare(key, gold.get(key), prediction.get(key),
                 medication="-", rows=rows, counts=counts, opts=opts)

    gold_meds = gold.get("medications") or []
    pred_meds = prediction.get("medications") or []
    pairs, only_gold, only_pred = align(gold_meds, pred_meds, positional=positional)

    for i, j in pairs:
        for key in ATTRIBUTES:
            _compare(key, get_attr(gold_meds[i], key), get_attr(pred_meds[j], key),
                     medication=f"g{i + 1}~p{j + 1}", rows=rows, counts=counts, opts=opts)
    for i in only_gold:
        for key in ATTRIBUTES:
            _compare(key, get_attr(gold_meds[i], key), None,
                     medication=f"g{i + 1}~-", rows=rows, counts=counts, opts=opts)
    for j in only_pred:
        for key in ATTRIBUTES:
            _compare(key, None, get_attr(pred_meds[j], key),
                     medication=f"-~p{j + 1}", rows=rows, counts=counts, opts=opts)

    med = Counter(tp=len(pairs), fn=len(only_gold), fp=len(only_pred))
    return counts, med, rows


def score_missing(gold, **opts):
    """A note with no usable output: every annotated slot becomes a false negative."""
    counts = defaultdict(Counter)
    rows = []
    for key in TOP_LEVEL_KEYS:
        _compare(key, gold.get(key), None,
                 medication="-", rows=rows, counts=counts, opts=opts)
    gold_meds = gold.get("medications") or []
    for i, medication in enumerate(gold_meds):
        for key in ATTRIBUTES:
            _compare(key, get_attr(medication, key), None,
                     medication=f"g{i + 1}~-", rows=rows, counts=counts, opts=opts)
    return counts, Counter(tp=0, fp=0, fn=len(gold_meds)), rows

### 4.4 The whole grid

`evaluate` scores every (model, strategy) over every annotated note. Per-note counts are
kept because the bootstrap resamples notes, not slots.

In [ ]:
def prf(counter):
    tp, fp, fn = counter["tp"], counter["fp"], counter["fn"]
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def evaluate(gold_by_id, records, *, skip_invalid=False, positional=False, **opts):
    by_run = defaultdict(dict)
    for record in records:
        if record["note_id"] in gold_by_id:
            by_run[(record["model"], record["strategy"])][record["note_id"]] = record

    out = {}
    for run, by_note in sorted(by_run.items()):
        per_note, per_key, disagreements = {}, defaultdict(Counter), []
        med_counts = Counter()
        invalid = []
        for note_id, gold in gold_by_id.items():
            record = by_note.get(note_id)
            valid = record is not None and record.get("status") == "ok"
            if not valid:
                invalid.append(note_id)
                if skip_invalid:
                    continue
                counts, med, rows = score_missing(gold, **opts)
            else:
                counts, med, rows = score_note(
                    gold, record.get("output"), positional=positional, **opts
                )
            for key, counter in counts.items():
                per_key[key] += counter
            med_counts += med
            per_note[note_id] = {
                "counts": sum((counts[k] for k in SCORED_KEYS), Counter()),
                "valid": valid,
                "n_gold_medications": len(gold.get("medications") or []),
            }
            for row in rows:
                disagreements.append({"model": run[0], "strategy": run[1],
                                      "note_id": note_id, "valid": valid, **row})
        out[run] = {
            "per_note": per_note,
            "per_key": dict(per_key),
            "medications": med_counts,
            "micro": sum((per_key[k] for k in SCORED_KEYS if k in per_key), Counter()),
            "invalid": invalid,
            "n_notes": len(per_note),
            "disagreements": disagreements,
        }
    return out


results = evaluate(gold_by_id, records)
RUNS = list(results)
MODELS = sorted({model for model, _ in RUNS})
STRATEGIES = sorted({strategy for _, strategy in RUNS})
print(len(RUNS), "runs:", MODELS, "x", STRATEGIES)

## 5. Headline table — errors counted as no output

`f1_lo` / `f1_hi` are the 2.5th and 97.5th percentiles of a bootstrap over the annotated
notes (`N_BOOTSTRAP` resamples, fixed seed). With 32 notes the interval is wide; that
width is the result, and any ranking inside it is not.

In [ ]:
NOTE_IDS = sorted(gold_by_id)
rng = np.random.default_rng(SEED)
BOOT_INDEX = rng.integers(0, len(NOTE_IDS), size=(N_BOOTSTRAP, len(NOTE_IDS)))


def note_matrix(run_result):
    """(n_notes, 3) array of tp/fp/fn per note, in NOTE_IDS order."""
    rows = []
    for note_id in NOTE_IDS:
        counter = run_result["per_note"].get(note_id, {}).get("counts", Counter())
        rows.append([counter["tp"], counter["fp"], counter["fn"]])
    return np.array(rows, dtype=float)


def bootstrap_f1(run_result):
    matrix = note_matrix(run_result)
    sums = matrix[BOOT_INDEX].sum(axis=1)          # (N_BOOTSTRAP, 3)
    tp, fp, fn = sums[:, 0], sums[:, 1], sums[:, 2]
    denom = 2 * tp + fp + fn
    return np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)


records_out = []
for run, result in results.items():
    precision, recall, f1 = prf(result["micro"])
    boot = bootstrap_f1(result)
    med_p, med_r, med_f1 = prf(result["medications"])
    records_out.append({
        "model": run[0], "strategy": run[1],
        "precision": precision, "recall": recall, "f1": f1,
        "f1_lo": np.percentile(boot, 2.5), "f1_hi": np.percentile(boot, 97.5),
        "tp": result["micro"]["tp"], "fp": result["micro"]["fp"],
        "fn": result["micro"]["fn"],
        "support": result["micro"]["tp"] + result["micro"]["fn"],
        "n_notes": result["n_notes"],
        "invalid_rate": len(result["invalid"]) / len(NOTE_IDS),
        "med_precision": med_p, "med_recall": med_r, "med_f1": med_f1,
    })

runs = (pd.DataFrame(records_out)
        .sort_values("f1", ascending=False)
        .reset_index(drop=True))
runs[["model", "strategy", "precision", "recall", "f1", "f1_lo", "f1_hi",
      "tp", "fp", "fn", "support", "n_notes", "invalid_rate"]]

In [ ]:
for metric in ["precision", "recall", "f1"]:
    print(f"\n=== {metric} ===")
    print(runs.pivot(index="model", columns="strategy", values=metric).round(3))

## 6. Medication detection, and the long span

Two things the attribute micro-average hides. **Detection** is how many medication
objects the model finds at all (the alignment of section 4.2); attribute scores are
conditional on it. **`medications_text`** is the verbatim section span, scored on its own
because a single whitespace difference loses the whole key.

In [ ]:
detection = runs.set_index(["model", "strategy"])[
    ["med_precision", "med_recall", "med_f1"]
].sort_values("med_f1", ascending=False)
detection.round(3)

In [ ]:
span_rows = []
for run, result in results.items():
    counter = result["per_key"].get(SPAN_KEY, Counter())
    precision, recall, f1 = prf(counter)
    span_rows.append({"model": run[0], "strategy": run[1],
                      "P": precision, "R": recall, "F1": f1,
                      "exact": counter["tp"], "wrong": counter["fn"]})
span_table = pd.DataFrame(span_rows).set_index(["model", "strategy"])
span_table.round(3) if SPAN_KEY in TOP_LEVEL_KEYS else "no medications_text key in schema"

## 7. Secondary views

* `valid_only` drops the failed notes entirely: quality *conditional* on producing
  parseable JSON. Read it with `invalid_rate` — high `f1_valid_only` with high
  `invalid_rate` is a model that is accurate when it answers and often does not.
* `casefold` ignores capitalisation: how much of the strict error is
  `albuterol sulfate` vs `Albuterol Sulfate`. This is also the number that decides the
  open tall-man question (`OxycoDONE`) — if the gap is large, the annotation rule is
  doing the work, not the model.
* `whitespace` additionally collapses runs of whitespace: formatting noise.
* `positional` is the old position-by-position comparison, kept as a lower bound; the
  gap to `f1_strict` is the cost of list misalignment.

In [ ]:
views = {
    "f1_valid_only": evaluate(gold_by_id, records, skip_invalid=True),
    "f1_casefold": evaluate(gold_by_id, records, casefold=True),
    "f1_whitespace": evaluate(gold_by_id, records, casefold=True, collapse_ws=True),
    "f1_positional": evaluate(gold_by_id, records, positional=True),
}


def f1_series(res, name):
    return pd.Series(
        {run: prf(result["micro"])[2] for run, result in res.items()}, name=name
    ).rename_axis(["model", "strategy"])


comparison = (
    runs.set_index(["model", "strategy"])[["f1", "invalid_rate"]]
    .rename(columns={"f1": "f1_strict"})
    .join([f1_series(res, name) for name, res in views.items()])
)
comparison["gap_errors"] = comparison.f1_valid_only - comparison.f1_strict
comparison["gap_casing"] = comparison.f1_casefold - comparison.f1_strict
comparison["gap_alignment"] = comparison.f1_strict - comparison.f1_positional
comparison.sort_values("f1_strict", ascending=False).round(3)

## 8. Is the difference real? Paired bootstrap

The strategies are run on the same 32 notes, so they are compared paired: resample the
notes, recompute both F1 scores on the same resample, look at the distribution of the
difference. `p_two_sided` is the fraction of resamples where the sign flips (doubled);
with this sample size expect it to be uninformative for small gaps — which is the point.

In [ ]:
def paired_bootstrap(run_a, run_b):
    fa, fb = bootstrap_f1(results[run_a]), bootstrap_f1(results[run_b])
    diff = fa - fb
    observed = prf(results[run_a]["micro"])[2] - prf(results[run_b]["micro"])[2]
    p = 2 * min((diff <= 0).mean(), (diff >= 0).mean())
    return {"a": f"{run_a[0]}/{run_a[1]}", "b": f"{run_b[0]}/{run_b[1]}",
            "delta_f1": observed,
            "lo": np.percentile(diff, 2.5), "hi": np.percentile(diff, 97.5),
            "p_two_sided": min(p, 1.0)}


pairs_rows = [
    paired_bootstrap((model, a), (model, b))
    for model in MODELS
    for i, a in enumerate(STRATEGIES) for b in STRATEGIES[i + 1:]
    if (model, a) in results and (model, b) in results
]
pd.DataFrame(pairs_rows).sort_values("delta_f1", ascending=False).round(3)

## 9. Breakdown per key

`support` (the number of annotated slots) is shown next to the scores: a key annotated
five times cannot support a claim about a model, and it weighs as much as
`active_substance` in any macro average.

In [ ]:
per_key_rows = []
for run, result in results.items():
    for key in ALL_KEYS:
        counter = result["per_key"].get(key, Counter())
        precision, recall, f1 = prf(counter)
        per_key_rows.append({
            "model": run[0], "strategy": run[1], "key": key,
            "precision": precision, "recall": recall, "f1": f1,
            "tp": counter["tp"], "fp": counter["fp"], "fn": counter["fn"],
            "support": counter["tp"] + counter["fn"],
        })
per_key = pd.DataFrame(per_key_rows)

per_key.pivot(index="key", columns=["model", "strategy"], values="f1") \
       .reindex(ALL_KEYS).round(3)

In [ ]:
per_key.groupby("key").agg(support=("support", "max"), mean_f1=("f1", "mean")) \
       .reindex(ALL_KEYS).sort_values("support", ascending=False).round(3)

## 10. Breakdown per note

In [ ]:
per_note_rows = []
for run, result in results.items():
    for note_id, entry in result["per_note"].items():
        precision, recall, f1 = prf(entry["counts"])
        per_note_rows.append({
            "model": run[0], "strategy": run[1], "note_id": note_id,
            "precision": precision, "recall": recall, "f1": f1,
            "valid": entry["valid"],
            "n_gold_medications": entry["n_gold_medications"],
        })
per_note = pd.DataFrame(per_note_rows)

per_note.groupby("note_id").agg(
    mean_f1=("f1", "mean"),
    failed_runs=("valid", lambda s: (~s).sum()),
    n_gold_medications=("n_gold_medications", "max"),
).sort_values("mean_f1").head(10).round(3)

## 11. Every disagreement, categorised

One row per slot where gold and prediction differ, across the whole grid, with the
category assigned in section 4.3. This is the material for the error analysis chapter;
the aggregate below says which failure mode dominates each model, and the full table
goes to CSV in section 13.

In [ ]:
disagreements = pd.DataFrame(
    [row for result in results.values() for row in result["disagreements"]]
)

(disagreements[disagreements.valid]
 .pivot_table(index=["model", "strategy"], columns="category",
              values="note_id", aggfunc="count")
 .fillna(0).astype(int))

In [ ]:
(disagreements[disagreements.valid]
 .pivot_table(index="key", columns="category", values="note_id", aggfunc="count")
 .reindex(ALL_KEYS).fillna(0).astype(int))

In [ ]:
# the worst note of the best run, slot by slot
MODEL, STRATEGY = runs.loc[0, "model"], runs.loc[0, "strategy"]
worst = per_note[(per_note.model == MODEL) & (per_note.strategy == STRATEGY)] \
    .sort_values("f1").iloc[0]
print(f"{MODEL} / {STRATEGY} — note {worst.note_id} (F1 {worst.f1:.3f})")

disagreements[(disagreements.model == MODEL)
              & (disagreements.strategy == STRATEGY)
              & (disagreements.note_id == worst.note_id)][
    ["medication", "key", "gold", "predicted", "category"]
]

## 12. Final tables — one row per key, P/R/F1 per model and prompting strategy

Keys as rows, `P | R | F1` inside each column block.

* keys that are never annotated (no gold value anywhere) are dropped — an all-zero row
  says nothing about the model, only that the attribute does not occur in this split;
* **Macro** = unweighted mean over the keys shown, so a rare key weighs as much as
  `active_substance`;
* **Micro** = counts of those keys pooled, which is the headline number of section 5;
* a `support` column is kept so no row can be read without its sample size;
* failed extractions are still counted as missing output.

`models=` / `strategies=` restrict the columns, which gives one table per model or one
per prompting strategy; the column level that becomes constant is dropped automatically.

In [ ]:
def metrics_table(results, *, models=None, strategies=None, keys=None,
                  with_support=True):
    selected = [
        run for run in results
        if (models is None or run[0] in models)
        and (strategies is None or run[1] in strategies)
    ]
    keys = keys or SCORED_KEYS
    keys = [k for k in keys
            if any(results[run]["per_key"].get(k, Counter())["tp"]
                   + results[run]["per_key"].get(k, Counter())["fn"]
                   for run in selected)]

    frame = {}
    for run in selected:
        per_key_counts = results[run]["per_key"]
        column = []
        for key in keys:
            column.append(prf(per_key_counts.get(key, Counter())))
        macro = tuple(np.mean([c[i] for c in column]) for i in range(3))
        micro = prf(sum((per_key_counts.get(k, Counter()) for k in keys), Counter()))
        for offset, metric in enumerate(["P", "R", "F1"]):
            frame[(run[0], run[1], metric)] = (
                [c[offset] for c in column] + [macro[offset], micro[offset]]
            )

    table = pd.DataFrame(frame, index=keys + ["Macro", "Micro"])
    table.columns = pd.MultiIndex.from_tuples(table.columns,
                                              names=["model", "strategy", "metric"])
    for level in ("model", "strategy"):
        if table.columns.get_level_values(level).nunique() == 1:
            table.columns = table.columns.droplevel(level)

    if with_support:
        support = []
        for key in keys:
            counters = [results[run]["per_key"].get(key, Counter()) for run in selected]
            support.append(max(c["tp"] + c["fn"] for c in counters) if counters else 0)
        table.insert(0, "support", support + [sum(support), sum(support)])
    return table


table_all = metrics_table(results)
table_all.round(2)

In [ ]:
for model in MODELS:
    print(f"\n=== {model} ===")
    print(metrics_table(results, models=[model]).round(2).to_string())

In [ ]:
for strategy in STRATEGIES:
    print(f"\n=== {strategy} ===")
    print(metrics_table(results, strategies=[strategy]).round(2).to_string())

In [ ]:
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(exist_ok=True)


def slug(name: str) -> str:
    return name.replace(":", "_").replace(".", "_").replace("/", "_")


headline = runs.set_index(["model", "strategy"])[
    ["precision", "recall", "f1", "f1_lo", "f1_hi", "invalid_rate", "med_f1"]
]
headline.columns = ["P", "R", "F1", "F1 lo", "F1 hi", "invalid", "med F1"]

exports = [
    ("summary",
     "Table 1: Exact-match Precision (P), Recall (R) and F1 per model and prompting "
     "strategy, over all annotated notes, with 95% bootstrap intervals over notes. "
     "Failed extractions counted as missing output; med F1 is medication detection.",
     headline),
    ("secondary_views",
     "Table 2: strict F1 against the valid-only, casefolded, whitespace-normalised and "
     "positional variants.",
     comparison.sort_values("f1_strict", ascending=False)),
]
exports += [
    (f"by_key_{slug(strategy)}",
     f"Table: exact-match results per key under {strategy} prompting.",
     metrics_table(results, strategies=[strategy]))
    for strategy in STRATEGIES
]
exports += [
    (f"by_key_{slug(model)}",
     f"Table: exact-match results per key for {model}.",
     metrics_table(results, models=[model]))
    for model in MODELS
]
exports.append(("appendix_full_grid",
                "Appendix table: every model and prompting strategy.", table_all))

tables = {caption: frame.round(3) for _, caption, frame in exports}

DOCX_PATH = TABLES_DIR / f"exact_match_tables_{RUN_ID}.docx"
try:
    metrics_to_docx(tables, str(DOCX_PATH),
                    title=f"Exact-match evaluation — run {RUN_ID}")
    print("wrote", DOCX_PATH)
except ImportError:
    print("python-docx not installed (uv add python-docx) — writing HTML only")

for name, caption, frame in exports:
    (TABLES_DIR / f"{name}.html").write_text(
        metrics_to_html(frame.round(3), caption=caption), encoding="utf-8"
    )

# LaTeX, if you ever need it:
# (TABLES_DIR / "exact_match_all.tex").write_text(
#     metrics_to_latex(table_all), encoding="utf-8")

print(sorted(p.name for p in TABLES_DIR.iterdir()))

## 13. Export the raw numbers

In [ ]:
runs.to_csv(OUTPUT_DIR / "metrics_per_run.csv", index=False)
per_key.to_csv(OUTPUT_DIR / "metrics_per_key.csv", index=False)
per_note.to_csv(OUTPUT_DIR / "metrics_per_note.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "metrics_secondary_views.csv")
detection.to_csv(OUTPUT_DIR / "metrics_medication_detection.csv")
pd.DataFrame(pairs_rows).to_csv(OUTPUT_DIR / "paired_bootstrap.csv", index=False)
disagreements.to_csv(OUTPUT_DIR / "disagreements.csv", index=False)
coverage.to_csv(OUTPUT_DIR / "coverage.csv")

(OUTPUT_DIR / "evaluation_settings.json").write_text(json.dumps({
    "run_id": RUN_ID, "split": SPLIT, "criterion": CRITERION,
    "seed": SEED, "n_bootstrap": N_BOOTSTRAP,
    "match_threshold": MATCH_THRESHOLD, "name_keys": list(NAME_KEYS),
    "scored_keys": SCORED_KEYS, "span_key": SPAN_KEY, "set_keys": sorted(SET_KEYS),
}, indent=2), encoding="utf-8")

print("written to", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))